# DM-SR Study — Reference Pack (CPU only)

Este notebook **não usa GPU** e não executa treino, avaliação ou meshing. Ele baixa o ZIP oficial DM-SR, valida o SHA-256 congelado e extrai somente a malha ground truth `study.ply` e quatro imagens RGB de teste igualmente espaçadas para avaliação qualitativa.

No Kaggle: deixe **Accelerator = None**, habilite **Internet** e use **Save Version → Save & Run All**. O único output necessário será `dmsr-study-reference-pack.zip`.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, zipfile

URL = 'https://www.dropbox.com/s/1k75m38vahizbp9/dmsr.zip?dl=1'
DATASET_SHA256 = '146ddd88d34cf8efc3357737871325f2ad1fe7584f7e20fc8ec75f5d3baed602'
GT_SHA256 = '8de1c71689452af874d661b610873a036fddf979b89c86704e5537561c1452fc'
ROOT = Path('/kaggle/temp/dmsr-reference-pack')
OUT = Path('/kaggle/working/dmsr-study-reference-pack')
ARCHIVE = ROOT / 'dmsr.zip'
ROOT.mkdir(parents=True, exist_ok=True)
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

subprocess.run([
    'curl','--fail','--location','--retry','3','--connect-timeout','30','--max-time','1800',
    '--output',str(ARCHIVE),URL
], check=True)
assert zipfile.is_zipfile(ARCHIVE), 'Download oficial não é ZIP'
actual = sha256(ARCHIVE)
assert actual == DATASET_SHA256, f'SHA do dataset divergente: {actual}'

members = [
    'dmsr/study/study.ply',
    'dmsr/study/test/rgbs/r_0000.png',
    'dmsr/study/test/rgbs/r_0033.png',
    'dmsr/study/test/rgbs/r_0066.png',
    'dmsr/study/test/rgbs/r_0099.png',
]
with zipfile.ZipFile(ARCHIVE) as z:
    available = set(z.namelist())
    missing = [m for m in members if m not in available]
    assert not missing, f'Arquivos ausentes no ZIP: {missing}'
    for member in members:
        target = OUT / Path(member).name
        with z.open(member) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

gt = OUT / 'study.ply'
gt_sha = sha256(gt)
assert gt_sha == GT_SHA256, f'SHA do study.ply divergente: {gt_sha}'
manifest = {
    'dataset_url': URL,
    'dataset_sha256': DATASET_SHA256,
    'study_ply_sha256': gt_sha,
    'study_ply_bytes': gt.stat().st_size,
    'rgb_views': [0, 33, 66, 99],
}
(OUT/'reference_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')

archive_out = shutil.make_archive('/kaggle/working/dmsr-study-reference-pack', 'zip', OUT)
ARCHIVE.unlink(missing_ok=True)
print(json.dumps(manifest, indent=2))
print('RESULTS:', archive_out)
